# Code 8.2.4: A block with RMSNorm, RoPE, and SwiGLU

A block with the three recent substitutions. It checks that the SwiGLU FFN with $`d_{\text{ff}} = \tfrac{8}{3} d`$ has $`8d^2`$ weights, that the block has $`12d^2`$ weights plus two RMSNorm gain vectors, and that it is still causal, then applies the parameter formula to GPT-3.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLU(nn.Module):                                # W_2 (SiLU(W_1 x) * W_3 x), no biases
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, d_ff, bias=False), nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

def rope(x, base=10000.0):
    """Rotary embedding for x of shape (B, h, n, d_k): rotate each pair of dimensions
    of the vector at position p by the angle p * omega_i (Section 6.6)."""
    n, dk = x.shape[-2], x.shape[-1]
    omega = base ** (-torch.arange(0, dk, 2, dtype=torch.float32) / dk)
    ang = torch.arange(n, dtype=torch.float32)[:, None] * omega        # (n, d_k/2)
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

class ModernBlock(nn.Module):                           # RMSNorm + RoPE attention + SwiGLU
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1, self.norm2 = nn.RMSNorm(d), nn.RMSNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)
        self.ffn, self.h = SwiGLU(d, d_ff), h
    def attn(self, x):
        B, n, d = x.shape
        q, k, v = (t.view(B, n, self.h, -1).transpose(1, 2) for t in self.qkv(x).split(d, -1))
        q, k = rope(q), rope(k)                         # positions enter through q and k only
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, n, d))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

d, h = 768, 12
d_ff = int(8 * d / 3)                                   # 2048
blk = ModernBlock(d, h, d_ff)
ffn = sum(p.numel() for p in blk.ffn.parameters())
total = sum(p.numel() for p in blk.parameters())
print(d_ff, ffn, 8 * d * d)                             # d_ff, SwiGLU weights, 8d^2
print(total, 12 * d * d, total - 12 * d * d)            # block, 12d^2, difference (two RMSNorm gains)
x = torch.randn(2, 10, d)
x2 = x.clone(); x2[:, 6:] = torch.randn(2, 4, d)
print(torch.allclose(blk(x)[:, :6], blk(x2)[:, :6], atol=1e-5))  # still causal?

# The same formula for GPT-3 (Brown et al. 2020): N = 96, d = 12,288, n_max = 2,048
N, d, V, n_max = 96, 12288, 50257, 2048
print(f"{(12 * N * d**2 + V * d + n_max * d) / 1e9:.1f}B")
# 2048 4718592 4718592
# 7079424 7077888 1536
# True
# 174.6B
